# Startup/Product Forecasting

Generate a forecasting dataset about startup and product survival using Show HN posts from the Hacker News BigQuery dataset. Questions focus on longevity (will this product still exist in X years?) and funding (will they reach Series B?). WebSearchLabeler verifies outcomes via web search.

In [7]:
%pip install python-dotenv pandas
%pip install -e ..

from IPython.display import clear_output
clear_output()

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

**GCP credentials**: BigQuery requires GCP credentials. Set `GOOGLE_APPLICATION_CREDENTIALS` or use default application credentials.

In [8]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## BigQuery configuration

Query filters Show HN and Launch HN posts from 2019–2021 so questions like "Will X still be around in 3 years?" have resolution dates in 2022–2024 (already in the past). WebSearchLabeler can verify survival.

Excludes generic OSS tooling (libraries, frameworks, parsers, CLI tools, etc.) to focus on products and startups.

In [9]:
from lightningrod import BigQuerySeedGenerator

HN_QUERY = """
SELECT
  CONCAT(
    'Title: ', COALESCE(title, ''),
    '\\n\\nContent: ', COALESCE(REGEXP_REPLACE(COALESCE(text, ''), r'<[^>]*>', ''), COALESCE(url, '')),
    '\\n\\nURL: ', COALESCE(url, '')
  ) AS content,
  TIMESTAMP_SECONDS(time) AS time
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND score > 100
  AND title IS NOT NULL
  AND url IS NOT NULL
  AND (title LIKE 'Show HN%' OR title LIKE 'Launch HN%')
  AND time >= UNIX_SECONDS(TIMESTAMP('2019-01-01'))
  AND time < UNIX_SECONDS(TIMESTAMP('2022-01-01'))
  AND NOT REGEXP_CONTAINS(LOWER(title), r'library|framework|parser|wrapper|emulator|boilerplate|starter (kit|template|project)|plugin for|extension for|api client|compiler|linter|formatter|cli tool|command[- ]?line')
ORDER BY time DESC
"""

seed_generator = BigQuerySeedGenerator(
    query=HN_QUERY,
    seed_text_column="content",
    date_column="time",
)

## Build the pipeline

ForwardLookingQuestionGenerator produces questions with prediction_date and date_close. WebSearchLabeler verifies survival via web search. No NewsContextGenerator — the seed contains the full post; use `drop_missing_context=False` in prepare.

In [10]:
INSTRUCTIONS = """
Generate binary forecasting questions about whether a product or startup from this Show HN post will survive or succeed.
Focus on: (1) longevity — will the product/company still exist in X years? (2) funding — will they reach the next stage?
Use the post title, content, and URL to identify the product. Questions must be forward-looking from the post date and verifiable via web search.
"""

EXAMPLES = [
    "Will this product still be around in 3 years?",
    "Will this startup still be operational in 2 years?",
    "Will this company raise Series A within 18 months?",
]

BAD_EXAMPLES = [
    "What technology does this use?",
    "When was this founded?",
    "Is this B2B or B2C?",
]

In [11]:
from lightningrod import (
    BinaryAnswerType,
    ForwardLookingQuestionGenerator,
    WebSearchLabeler,
    QuestionRenderer,
    QuestionPipeline,
    FilterCriteria,
)

answer_type = BinaryAnswerType()

pipeline_filter = FilterCriteria(
    rubric="The question must be about a product or startup (company, SaaS, marketplace, commercial offering) — not about generic open source libraries, frameworks, parsers, CLI tools, or developer utilities.",
    min_score=0.6,
)

pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=ForwardLookingQuestionGenerator(
        instructions=INSTRUCTIONS,
        examples=EXAMPLES,
        bad_examples=BAD_EXAMPLES,
        answer_type=answer_type,
        questions_per_seed=2,
        filter_=pipeline_filter
    ),
    labeler=WebSearchLabeler(
        answer_type=answer_type,
        confidence_threshold=0.5,
    ),
    renderer=QuestionRenderer(answer_type=answer_type),
)

> Note: Processing can take several minutes (BigQuery fetch, question generation, web search labeling).

## Run the pipeline

In [12]:
dataset = lr.transforms.run(pipeline, max_questions=500, name="Startup forecasting")
samples = dataset.download()

pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Warning                                                                                                     │
│                                                                                                                 │
│  Estimated cost ($78.52) exceeds current balance ($49.76). Consider adding credits before running this job.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

1060 samples (19.0% valid)


## Prepare the dataset

Filter valid samples, deduplicate, and split into train/test. Use `drop_missing_context=False` (no NewsContextGenerator) and `days_to_resolution_range=(365, None)` to keep 1+ year horizons.

In [17]:
from lightningrod.training import prepare_for_training

train, test = prepare_for_training(
    samples,
    answer_type,
    test_size=0.2,
    split_strategy="temporal",
    include_assistant=True,
    filter_leaky_train=False,
    days_to_resolution_range=(365, None),
    drop_missing_context=False,
)

for name, data in [("Train", train), ("Test", test)]:
    if data:
        yes_count = sum(s["label"] or 0 for s in data)
        print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")
    else:
        print(f"{name}: 0 rows")

Train: 153 rows, 48.4% yes
Test: 39 rows, 53.8% yes


## Results

In [14]:
def _display_head(data, name, n=5):
    if not data:
        print(f"{name}: no rows")
        return
    df = pd.DataFrame(data[:n])
    # Prevent question_text truncation in pandas DataFrame display
    pd.set_option('display.max_colwidth', None)
    cols = ["question_text", "prediction_date", "date_close", "resolution_date", "label", "label_confidence"]
    display_cols = [c for c in cols if c in df.columns]
    print(f"{name} (head):")
    display(df[display_cols])

_display_head(train, "Train")
_display_head(test, "Test")

Train (head):


,question_text,prediction_date,date_close,resolution_date,label,label_confidence
0,"Will WebsiteVoice, LLC or its parent company have officially announced a venture capital funding round (including Seed, Series A, or later) of at least $1 million USD by July 1, 2022?",2019-01-07T02:56:45,2022-07-01T00:00:00,2022-07-01T00:00:00,0.0,0.9
1,"Will the company or project behind Polar (getpolarized.io) announce a venture capital funding round (Seed, Series A, or later) of at least $1 million USD by January 1, 2022?",2019-01-09T15:43:35,2022-01-01T00:00:00,2022-01-01T00:00:00,0.0,1.0
2,"Will Mikado Software, the creator of the 'workstation' project, be listed as an active company on the UK Companies House register as of January 1, 2025?",2019-01-10T22:52:58,2025-01-05T00:00:00,2025-01-01T00:00:00,1.0,1.0
3,"Will the website heyfromthefuture.com still be operational on July 14, 2025?",2019-01-14T12:48:00,2025-07-14T00:00:00,2024-12-31T00:00:00,0.0,1.0
4,"Will the 'Plain Freelance Contract' project, as launched at plainfreelancecontract.com, still be online and publicly accessible on January 16, 2024?",2019-01-15T14:32:50,2024-01-16T00:00:00,2024-01-16T00:00:00,1.0,1.0


Test (head):


,question_text,prediction_date,date_close,resolution_date,label,label_confidence
0,"Will the website Screentop.gg be operational and accessible for play on January 10, 2023?",2020-01-10T21:24:06,2023-01-10T00:00:00,2023-01-10T00:00:00,1.0,1.0
1,"Will the domain podnami.com resolve to an active, operational website dedicated to technology podcast discovery on January 12, 2023?",2020-01-12T16:35:58,2023-01-12T00:00:00,2023-01-12T00:00:00,0.0,0.9
2,"Will the Diary Email service (diaryemail.com) still be operational as a live website on January 14, 2023?",2020-01-14T14:37:40,2023-01-14T00:00:00,2023-01-14T00:00:00,1.0,0.9
3,"Will vesoft-inc, the company behind Nebula Graph, announce a Series B funding round by December 31, 2022?",2020-01-15T02:27:06,2022-12-31T00:00:00,2022-12-31T00:00:00,0.0,1.0
4,"Will the 'Screenshot Hero' app by Asad Memon be available for download on the official Apple App Store on January 1, 2023?",2020-01-16T15:24:38,2023-01-01T00:00:00,2023-01-01T00:00:00,0.0,0.9


## Uploading the dataset to HuggingFace

Once we have a training-ready dataset, we can push it to Hugging Face for sharing or downstream use.

In [15]:
%pip install datasets -q

from datasets import Dataset, DatasetDict
from lightningrod.utils import config

dataset = DatasetDict({
    "train": Dataset.from_list(train),
    "test": Dataset.from_list(test),
})
print(f"Train: {len(dataset['train'])} rows, Test: {len(dataset['test'])} rows")
print("Columns:", dataset["train"].column_names[:8], "...")

DATASET_PATH = f"{config.get_config_value('HF_USERNAME')}/startup-forecasting-demo"
dataset.push_to_hub(DATASET_PATH, token=config.get_config_value("HF_ACCESS_TOKEN"))


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Train: 153 rows, Test: 39 rows
Columns: ['question_text', 'date_close', 'event_date', 'resolution_criteria', 'prediction_date', 'label', 'answer_type', 'label_confidence'] ...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/933 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/bart/startup-forecasting-demo/commit/2678df1d6d456022b728a9e086896fc0301a34dd', commit_message='Upload dataset', commit_description='', oid='2678df1d6d456022b728a9e086896fc0301a34dd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bart/startup-forecasting-demo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bart/startup-forecasting-demo'), pr_revision=None, pr_num=None)